## Ablation: Drop Signal Binaries + Network Features

Remove 10 features (7 sig_* binaries, max_jaccard, pct_shared_npis, pct_multi_practice_overlap) and rerun PU bagging to test whether the remaining 28 features produce equivalent performance.

In [32]:
# ablation feature set -- drop 7 sig_* binaries + 3 network features
drop_features = [
    "sig_ramp", "sig_no_prior", "sig_no_treat", "sig_mbr_multi_tin",
    "sig_billing_conc", "sig_churn", "sig_tin_network",
    "max_jaccard", "pct_shared_npis", "pct_multi_practice_overlap",
]

abl_continuous = [f for f in continuous_features if f not in drop_features]
abl_features = abl_continuous + [f for f in binary_features if f not in drop_features]
print(f"Ablation features: {len(abl_features)} ({len(abl_continuous)} continuous + {len(abl_features) - len(abl_continuous)} binary)")
print(f"Dropped: {drop_features}")

Ablation features: 28 (28 continuous + 0 binary)
Dropped: ['sig_ramp', 'sig_no_prior', 'sig_no_treat', 'sig_mbr_multi_tin', 'sig_billing_conc', 'sig_churn', 'sig_tin_network', 'max_jaccard', 'pct_shared_npis', 'pct_multi_practice_overlap']


In [33]:
# prep ablation X -- same log-transform + scale as original
skew_to_transform_abl = [f for f in skew_features if f in abl_continuous]

X_abl_raw = df[abl_features].copy()
for f in skew_to_transform_abl:
    X_abl_raw[f] = np.log1p(X_abl_raw[f].clip(lower = 0))

scaler_abl = StandardScaler()
X_abl_raw[abl_continuous] = scaler_abl.fit_transform(X_abl_raw[abl_continuous])

X_abl = X_abl_raw.values
print(f"X_abl shape: {X_abl.shape}")

X_abl shape: (7254, 28)


In [34]:
# ablation PU bagging -- RF only
K_abl = 100
np.random.seed(42)

abl_rf_scores = np.zeros((K_abl, len(y)))

for k in range(K_abl):
    neg_sample = np.random.choice(unl_idx, size = n_pos, replace = False)
    train_idx = np.concatenate([pos_idx, neg_sample])
    X_train = X_abl[train_idx]
    y_train = np.concatenate([np.ones(n_pos), np.zeros(n_pos)])

    rf = RandomForestClassifier(n_estimators = 200, max_depth = 6, min_samples_leaf = 5, random_state = k)
    rf.fit(X_train, y_train)
    abl_rf_scores[k] = rf.predict_proba(X_abl)[:, 1]

    if (k + 1) % 25 == 0:
        print(f"  Iteration {k + 1}/{K_abl} complete")

abl_vendor_score = abl_rf_scores.mean(axis = 0)
print(f"\nAblation RF scores: mean = {abl_vendor_score.mean():.4f}, std = {abl_vendor_score.std():.4f}")

  Iteration 25/100 complete
  Iteration 50/100 complete
  Iteration 75/100 complete
  Iteration 100/100 complete

Ablation RF scores: mean = 0.2652, std = 0.1684


In [35]:
# ablation 5-fold CV recall@200
skf_abl = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)
abl_holdout_recalls = []

for fold, (train_pos_rel, test_pos_rel) in enumerate(skf_abl.split(pos_idx, np.ones(n_pos))):
    train_pos = pos_idx[train_pos_rel]
    test_pos = pos_idx[test_pos_rel]
    n_train_pos = len(train_pos)

    fold_rf_scores = np.zeros((50, len(y)))
    for k in range(50):
        neg_sample = np.random.choice(unl_idx, size = n_train_pos, replace = False)
        train_idx = np.concatenate([train_pos, neg_sample])
        X_train = X_abl[train_idx]
        y_train = np.concatenate([np.ones(n_train_pos), np.zeros(n_train_pos)])

        rf = RandomForestClassifier(n_estimators = 200, max_depth = 6, min_samples_leaf = 5,
                                    random_state = fold * 100 + k)
        rf.fit(X_train, y_train)
        fold_rf_scores[k] = rf.predict_proba(X_abl)[:, 1]

    fold_avg = fold_rf_scores.mean(axis = 0)
    top_200 = np.argsort(-fold_avg)[:200]
    recall = len(set(test_pos) & set(top_200)) / len(test_pos)
    abl_holdout_recalls.append(recall)
    print(f"  Fold {fold + 1}: RF recall@200 = {recall:.3f}")

print(f"\nAblation RF mean recall@200: {np.mean(abl_holdout_recalls):.3f} +/- {np.std(abl_holdout_recalls):.3f}")

  Fold 1: RF recall@200 = 0.583
  Fold 2: RF recall@200 = 0.750
  Fold 3: RF recall@200 = 0.792
  Fold 4: RF recall@200 = 0.708
  Fold 5: RF recall@200 = 0.625

Ablation RF mean recall@200: 0.692 +/- 0.077


In [36]:
# ablation vs original comparison
print("=== Ablation vs Original (RF) ===\n")

print("Recall@200 (5-fold CV):")
print(f"  Original (38 features): {np.mean(holdout_recalls_rf):.3f} +/- {np.std(holdout_recalls_rf):.3f}")
print(f"  Ablation (28 features): {np.mean(abl_holdout_recalls):.3f} +/- {np.std(abl_holdout_recalls):.3f}")

print("\nPrecision@k:")
for k in [100, 200, 300, 500]:
    orig_top = np.argsort(-rf_vendor_score)[:k]
    abl_top = np.argsort(-abl_vendor_score)[:k]
    orig_prec = y[orig_top].sum() / k
    abl_prec = y[abl_top].sum() / k
    print(f"  Top {k}: Original = {orig_prec:.1%}, Ablation = {abl_prec:.1%}")

print("\nScore distribution:")
for label, scores in [("Original", rf_vendor_score), ("Ablation", abl_vendor_score)]:
    c_scores = scores[y == 1]
    nc_scores = scores[y == 0]
    print(f"  {label} Cadence:     median = {np.median(c_scores):.4f}, IQR = [{np.percentile(c_scores, 25):.4f}, {np.percentile(c_scores, 75):.4f}]")
    print(f"  {label} Non-Cadence: median = {np.median(nc_scores):.4f}, IQR = [{np.percentile(nc_scores, 25):.4f}, {np.percentile(nc_scores, 75):.4f}]")

rho, _ = stats.spearmanr(rf_vendor_score, abl_vendor_score)
print(f"\nSpearman correlation (original vs ablation scores): {rho:.4f}")

=== Ablation vs Original (RF) ===

Recall@200 (5-fold CV):
  Original (38 features): 0.692 +/- 0.077
  Ablation (28 features): 0.692 +/- 0.077

Precision@k:
  Top 100: Original = 75.0%, Ablation = 74.0%
  Top 200: Original = 45.5%, Ablation = 45.5%
  Top 300: Original = 33.0%, Ablation = 33.3%
  Top 500: Original = 21.8%, Ablation = 21.8%

Score distribution:
  Original Cadence:     median = 0.9014, IQR = [0.6767, 0.9716]
  Original Non-Cadence: median = 0.2327, IQR = [0.1471, 0.3439]
  Ablation Cadence:     median = 0.9045, IQR = [0.6825, 0.9731]
  Ablation Non-Cadence: median = 0.2286, IQR = [0.1414, 0.3408]

Spearman correlation (original vs ablation scores): 0.9980


# Cadence Vendor-Involvement Baseline

PU learning model: 120 Cadence TINs as labeled positives, 7,134 unlabeled. 38 features from RPM claims (M&R FFS), tre_membership, and fraud_scores. Score all 7,254 TINs on vendor-involvement probability.

In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from snowflake.sqlalchemy import URL
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from sklearn.model_selection import StratifiedKFold
import warnings
warnings.filterwarnings("ignore")

engine = create_engine(URL(
    account = "UHG-UHGDWAAS",
    user = "KHANG.NGUYEN@UHC.COM",
    authenticator = "externalbrowser",
    role = "AZU_SDRP_VING_PRD_DEVELOPER_ROLE",
    warehouse = "VING_PRD_MNR_HCE_DATAINFRA_WH",
    database = "VING_PRD_TREND_DB",
    schema = "TMP_1M"
))

df = pd.read_sql("select * from tmp_1m.knd_rpm_cadence_baseline", engine)
df.columns = df.columns.str.lower()
print(f"Shape: {df.shape}")
print(f"Cadence: {df['is_cadence'].sum()}, Non-Cadence: {(df['is_cadence'] == 0).sum()}")

c:\Users\knguy139\Documents\Projects\.venv\Lib\site-packages\snowflake\connector\vendored\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(
 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://login.microsoftonline.com/db05faca-c82a-4b9d-b9c5-0f64b6755421/saml2?SAMLRequest=lVJdb9sgFP0rFnu2wc43SlJlybKlatKocSptLxMBnKBgcAHXbX99sZ1I3UMr7QEJwTn3nHvPHd%2B85DJ45sYKrSYgjhAIuKKaCXWcgH26DIcgsI4oRqRWfAJeuQU307EluSzwrHQn9cCfSm5d4Aspi%2BuPCSiNwppYYbEiObfYUbybre9wEiFMrOXGeTlwoTArvNbJuQJDWFVVVHUibY4wQQhBNIIeVUO%2BgQ8SxdcahdFOUy2vlBff0ycSMUTdWsIjvML2QvwuVDuCr1QOLcjiX2m6Dbf3uxQEs2t3c61smXOz4%2BZZUL5%2FuGsNWO%2BgPB1Df1hFiP17IpFVusokOXOq86J0vmbkbzDjDEp9FH5Sq8UEFGfBOu78ez10umT75XmVueoNbW6Hu5878iPbP27u3wh6yqt5erteUxA8XnNN6lxX1pZ8peo0nX9CST9EozCJ07iDux2MBtEgHvwBwcKnKRRxDfNqufER5YIabXXmtJJC8cYlO6BeRigJ6TAhYfcwYuFhRHshyvrdQ3%2FQ63WTGNaZJaDdG9wYMdP%2Fm8YYfuReFnDjM1kttloK%2BhostcmJ%2BzyyOIqbF8HCrIFinhMhZ4wZbq2PTkpdzQ0nzu%2B5MyUHcNqq%2Frvp03c%3D&RelayState=ver%3A3-hint%3A260092392366162-ETMsDgAAAaDENMTFABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEJMiSwgMx2Nb3Ub0ly2HBUcAAACQ5GlLxmnKKSOI8I24u5BKhm

In [26]:
# feature definitions
id_cols = ["prov_tin", "sample_full_nm", "hospital_group", "tadm_prov_spec_cat",
           "provider_state", "provider_zip", "is_cadence"]

continuous_features = [
    "avg_member_age", "std_member_age", "pct_female", "pct_htn", "pct_dm", "pct_hf", "pct_invalid_dx",
    "avg_plan_tenure_months",
    "total_mbrs", "months_active", "avg_rpm_tenure_months", "single_month_churn_rate", "peak_mom_growth_pct",
    "pct_mgmt_lines", "pct_device_lines", "pct_setup_lines", "pct_99458", "proc_hhi",
    "avg_allowed_per_line", "avg_lines_per_mbr", "top_dom_pct", "weekend_pct", "avg_submission_lag",
    "distinct_srvc_npis", "mbrs_per_srvc_npi", "pct_srvc_ne_bil",
    "pct_no_prior_rel", "pct_no_treatment_mgmt", "pct_multi_practice_overlap", "max_jaccard", "pct_shared_npis",
]

binary_features = [
    "sig_ramp", "sig_no_prior", "sig_no_treat", "sig_mbr_multi_tin",
    "sig_billing_conc", "sig_churn", "sig_tin_network",
]

all_features = continuous_features + binary_features

null_counts = df[all_features].isnull().sum()
print("Features with nulls:")
print(null_counts[null_counts > 0])

Features with nulls:
Series([], dtype: int64)


In [27]:
# impute nulls
df["std_member_age"] = df["std_member_age"].fillna(0)
df["avg_plan_tenure_months"] = df["avg_plan_tenure_months"].fillna(df["avg_plan_tenure_months"].median())
df["pct_srvc_ne_bil"] = df["pct_srvc_ne_bil"].fillna(0)
df["mbrs_per_srvc_npi"] = df["mbrs_per_srvc_npi"].fillna(0)

assert df[all_features].isnull().sum().sum() == 0, "Nulls remain after imputation"
print("Nulls cleared.")

Nulls cleared.


In [28]:
# correlation filter on continuous features -- drop one from pairs with |r| > 0.90
corr = df[continuous_features].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k = 1).astype(bool))
high_corr_pairs = [(col, upper[col].idxmax(), upper[col].max())
                   for col in upper.columns if upper[col].max() > 0.90]

print("High-correlation pairs (|r| > 0.90):")
for c1, c2, r in sorted(high_corr_pairs, key = lambda x: -x[2]):
    print(f"  {c1} <-> {c2}: {r:.3f}")

to_drop = set()
for c1, c2, r in high_corr_pairs:
    to_drop.add(c2)

print(f"\nDropping {len(to_drop)} features: {to_drop}")
model_continuous = [f for f in continuous_features if f not in to_drop]
model_features = model_continuous + binary_features
print(f"Remaining features: {len(model_features)} ({len(model_continuous)} continuous + {len(binary_features)} binary)")

High-correlation pairs (|r| > 0.90):

Dropping 0 features: set()
Remaining features: 38 (31 continuous + 7 binary)


In [29]:
# log-transform right-skewed continuous features, then scale
skew_features = ["total_mbrs", "peak_mom_growth_pct", "distinct_srvc_npis",
                 "mbrs_per_srvc_npi", "avg_lines_per_mbr", "avg_submission_lag"]
skew_to_transform = [f for f in skew_features if f in model_continuous]

X_raw = df[model_features].copy()
for f in skew_to_transform:
    X_raw[f] = np.log1p(X_raw[f].clip(lower = 0))

scaler = StandardScaler()
X_raw[model_continuous] = scaler.fit_transform(X_raw[model_continuous])

X = X_raw.values
y = df["is_cadence"].values

print(f"X shape: {X.shape}, y positives: {y.sum()}, y negatives: {(y == 0).sum()}")

X shape: (7254, 38), y positives: 120, y negatives: 7134


## Descriptive Baseline

Cadence vs population: Cohen's d, Mann-Whitney U with BH-FDR correction.

In [30]:
# descriptive stats + statistical testing on RAW (pre-transform) features
cadence = df[df["is_cadence"] == 1]
non_cadence = df[df["is_cadence"] == 0]

results = []
for f in all_features:
    c_vals = cadence[f].dropna()
    nc_vals = non_cadence[f].dropna()

    c_med, c_q1, c_q3 = c_vals.median(), c_vals.quantile(0.25), c_vals.quantile(0.75)
    nc_med, nc_q1, nc_q3 = nc_vals.median(), nc_vals.quantile(0.25), nc_vals.quantile(0.75)

    pooled_std = np.sqrt(((len(c_vals) - 1) * c_vals.std()**2 + (len(nc_vals) - 1) * nc_vals.std()**2)
                         / (len(c_vals) + len(nc_vals) - 2))
    d = (c_vals.mean() - nc_vals.mean()) / pooled_std if pooled_std > 0 else 0

    u_stat, p_val = stats.mannwhitneyu(c_vals, nc_vals, alternative = "two-sided")

    results.append({
        "feature": f,
        "cadence_median": c_med, "cadence_iqr": f"{c_q1:.3f}-{c_q3:.3f}",
        "pop_median": nc_med, "pop_iqr": f"{nc_q1:.3f}-{nc_q3:.3f}",
        "cohens_d": d, "abs_d": abs(d),
        "mw_p": p_val,
    })

baseline = (
    pd.DataFrame(results)
    .assign(
        fdr_p = lambda x: stats.false_discovery_control(x["mw_p"], method = "bh"),
        significant = lambda x: x["fdr_p"] < 0.05,
        direction = lambda x: np.where(x["cohens_d"] > 0, "Cadence higher", "Cadence lower"),
    )
    .sort_values("abs_d", ascending = False)
)

print(f"Significant features (FDR < 0.05): {baseline['significant'].sum()} of {len(baseline)}")
print()
print(baseline[["feature", "cadence_median", "pop_median", "cohens_d", "fdr_p", "significant", "direction"]]
      .head(20).to_string(index = False))

Significant features (FDR < 0.05): 23 of 38

               feature  cadence_median  pop_median  cohens_d        fdr_p  significant      direction
    distinct_srvc_npis        6.000000    1.000000  1.199190 3.248385e-34         True Cadence higher
           top_dom_pct        0.215251    0.435205 -0.750937 6.653815e-22         True  Cadence lower
      pct_device_lines        0.516191    0.325565  0.683390 4.545951e-23         True Cadence higher
        pct_mgmt_lines        0.399776    0.624857 -0.665059 2.988626e-23         True  Cadence lower
             pct_99458        0.058032    0.145832 -0.630304 3.099020e-06         True  Cadence lower
                pct_hf        0.126330    0.000000  0.613031 1.927638e-36         True Cadence higher
        pct_invalid_dx        0.012652    0.029412 -0.480811 5.005636e-03         True  Cadence lower
        std_member_age        8.224533    6.461290  0.478373 7.833110e-10         True Cadence higher
avg_plan_tenure_months       62.32452

## PU Bagging Model

K=100 iterations. Each: 120 positives + 120 sampled unlabeled. Test both LR and RF as base learners.

In [ ]:
K = 100
np.random.seed(42)

pos_idx = np.where(y == 1)[0]
unl_idx = np.where(y == 0)[0]
n_pos = len(pos_idx)

lr_scores = np.zeros((K, len(y)))
rf_scores = np.zeros((K, len(y)))
lr_coefs = np.zeros((K, X.shape[1]))
rf_importances = np.zeros((K, X.shape[1]))

for k in range(K):
    neg_sample = np.random.choice(unl_idx, size = n_pos, replace = False)
    train_idx = np.concatenate([pos_idx, neg_sample])
    X_train = X[train_idx]
    y_train = np.concatenate([np.ones(n_pos), np.zeros(n_pos)])

    lr = LogisticRegression(penalty = "l2", C = 1.0, max_iter = 1000, random_state = k)
    lr.fit(X_train, y_train)
    lr_scores[k] = lr.predict_proba(X)[:, 1]
    lr_coefs[k] = lr.coef_[0]

    rf = RandomForestClassifier(n_estimators = 200, max_depth = 6, min_samples_leaf = 5, random_state = k)
    rf.fit(X_train, y_train)
    rf_scores[k] = rf.predict_proba(X)[:, 1]
    rf_importances[k] = rf.feature_importances_

    if (k + 1) % 25 == 0:
        print(f"  Iteration {k + 1}/{K} complete")

lr_vendor_score = lr_scores.mean(axis = 0)
rf_vendor_score = rf_scores.mean(axis = 0)
print(f"\nLR scores: mean = {lr_vendor_score.mean():.4f}, std = {lr_vendor_score.std():.4f}")
print(f"RF scores: mean = {rf_vendor_score.mean():.4f}, std = {rf_vendor_score.std():.4f}")

  Iteration 25/100 complete
  Iteration 50/100 complete
  Iteration 75/100 complete
  Iteration 100/100 complete

LR scores: mean=0.2406, std=0.2649
RF scores: mean=0.2682, std=0.1657


In [31]:
# feature importance summary
importance = (
    pd.DataFrame({
        "feature": model_features,
        "lr_coef_mean": lr_coefs.mean(axis = 0),
        "lr_coef_abs": np.abs(lr_coefs.mean(axis = 0)),
        "rf_importance": rf_importances.mean(axis = 0),
    })
    .assign(
        lr_rank = lambda x: x["lr_coef_abs"].rank(ascending = False).astype(int),
        rf_rank = lambda x: x["rf_importance"].rank(ascending = False).astype(int),
    )
    .sort_values("rf_importance", ascending = False)
)

print("Top 15 features by RF importance:")
print(importance[["feature", "lr_coef_mean", "lr_rank", "rf_importance", "rf_rank"]]
      .head(50).to_string(index = False))

Top 15 features by RF importance:
                   feature  lr_coef_mean  lr_rank  rf_importance  rf_rank
            pct_mgmt_lines     -0.486701       11       0.129441        1
               top_dom_pct     -0.693885        6       0.107632        2
                    pct_hf      0.251377       25       0.095220        3
          pct_device_lines      0.367560       19       0.086925        4
                 pct_99458     -0.407575       13       0.066844        5
        distinct_srvc_npis      0.889923        3       0.061778        6
        avg_submission_lag      0.229975       26       0.060855        7
     pct_no_treatment_mgmt     -0.324370       22       0.048097        8
           pct_setup_lines      0.268185       24       0.035535        9
                  proc_hhi      0.786300        5       0.033958       10
               weekend_pct      0.521854        9       0.029885       11
            pct_invalid_dx     -1.020227        2       0.021153       12
    

## Evaluation

Held-out positive recall, precision at top-k, score distribution, stability.

In [ ]:
# held-out positive recall: 5-fold CV
skf = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)
holdout_recalls_lr = []
holdout_recalls_rf = []

for fold, (train_pos_rel, test_pos_rel) in enumerate(skf.split(pos_idx, np.ones(n_pos))):
    train_pos = pos_idx[train_pos_rel]
    test_pos = pos_idx[test_pos_rel]
    n_train_pos = len(train_pos)

    fold_lr_scores = np.zeros((50, len(y)))
    fold_rf_scores = np.zeros((50, len(y)))

    for k in range(50):
        neg_sample = np.random.choice(unl_idx, size = n_train_pos, replace = False)
        train_idx = np.concatenate([train_pos, neg_sample])
        X_train = X[train_idx]
        y_train = np.concatenate([np.ones(n_train_pos), np.zeros(n_train_pos)])

        lr = LogisticRegression(penalty = "l2", C = 1.0, max_iter = 1000, random_state = fold * 100 + k)
        lr.fit(X_train, y_train)
        fold_lr_scores[k] = lr.predict_proba(X)[:, 1]

        rf = RandomForestClassifier(n_estimators = 200, max_depth = 6, min_samples_leaf = 5,
                                    random_state = fold * 100 + k)
        rf.fit(X_train, y_train)
        fold_rf_scores[k] = rf.predict_proba(X)[:, 1]

    fold_lr_avg = fold_lr_scores.mean(axis = 0)
    fold_rf_avg = fold_rf_scores.mean(axis = 0)

    top_200_lr = np.argsort(-fold_lr_avg)[:200]
    top_200_rf = np.argsort(-fold_rf_avg)[:200]
    recall_lr = len(set(test_pos) & set(top_200_lr)) / len(test_pos)
    recall_rf = len(set(test_pos) & set(top_200_rf)) / len(test_pos)
    holdout_recalls_lr.append(recall_lr)
    holdout_recalls_rf.append(recall_rf)
    print(f"  Fold {fold + 1}: LR recall@200={recall_lr:.3f}, RF recall@200={recall_rf:.3f}")

print(f"\nLR mean recall@200: {np.mean(holdout_recalls_lr):.3f} +/- {np.std(holdout_recalls_lr):.3f}")
print(f"RF mean recall@200: {np.mean(holdout_recalls_rf):.3f} +/- {np.std(holdout_recalls_rf):.3f}")

  Fold 1: LR recall@200=0.375, RF recall@200=0.583
  Fold 2: LR recall@200=0.375, RF recall@200=0.750
  Fold 3: LR recall@200=0.417, RF recall@200=0.792
  Fold 4: LR recall@200=0.542, RF recall@200=0.708
  Fold 5: LR recall@200=0.500, RF recall@200=0.625

LR mean recall@200: 0.442 +/- 0.068
RF mean recall@200: 0.692 +/- 0.077


In [ ]:
# precision at top-k + score distribution
for label, scores in [("LR", lr_vendor_score), ("RF", rf_vendor_score)]:
    print(f"\n{label} precision at top-k:")
    for k in [100, 200, 300, 500]:
        top_k = np.argsort(-scores)[:k]
        cadence_in_top = y[top_k].sum()
        precision = cadence_in_top / k
        print(f"  Top {k}: {int(cadence_in_top)}/{k} Cadence ({precision:.1%}), base rate = {y.mean():.1%}")

for label, scores in [("LR", lr_vendor_score), ("RF", rf_vendor_score)]:
    cadence_scores = scores[y == 1]
    non_cadence_scores = scores[y == 0]
    print(f"\n{label} score distribution:")
    print(f"  Cadence:     median={np.median(cadence_scores):.4f}, "
          f"Q1={np.percentile(cadence_scores, 25):.4f}, Q3={np.percentile(cadence_scores, 75):.4f}")
    print(f"  Non-Cadence: median={np.median(non_cadence_scores):.4f}, "
          f"Q1={np.percentile(non_cadence_scores, 25):.4f}, Q3={np.percentile(non_cadence_scores, 75):.4f}")


LR precision at top-k:
  Top 100: 30/100 Cadence (30.0%), base rate = 1.7%
  Top 200: 55/200 Cadence (27.5%), base rate = 1.7%
  Top 300: 70/300 Cadence (23.3%), base rate = 1.7%
  Top 500: 90/500 Cadence (18.0%), base rate = 1.7%

RF precision at top-k:
  Top 100: 75/100 Cadence (75.0%), base rate = 1.7%
  Top 200: 91/200 Cadence (45.5%), base rate = 1.7%
  Top 300: 99/300 Cadence (33.0%), base rate = 1.7%
  Top 500: 109/500 Cadence (21.8%), base rate = 1.7%

LR score distribution:
  Cadence:     median=0.8988, Q1=0.7805, Q3=0.9499
  Non-Cadence: median=0.1245, Q1=0.0330, Q3=0.3521

RF score distribution:
  Cadence:     median=0.9014, Q1=0.6767, Q3=0.9716
  Non-Cadence: median=0.2327, Q1=0.1471, Q3=0.3439


In [ ]:
# stability test
np.random.seed(999)
lr_scores_2 = np.zeros((K, len(y)))
rf_scores_2 = np.zeros((K, len(y)))

for k in range(K):
    neg_sample = np.random.choice(unl_idx, size = n_pos, replace = False)
    train_idx = np.concatenate([pos_idx, neg_sample])
    X_train = X[train_idx]
    y_train = np.concatenate([np.ones(n_pos), np.zeros(n_pos)])

    lr = LogisticRegression(penalty = "l2", C = 1.0, max_iter = 1000, random_state = 5000 + k)
    lr.fit(X_train, y_train)
    lr_scores_2[k] = lr.predict_proba(X)[:, 1]

    rf = RandomForestClassifier(n_estimators = 200, max_depth = 6, min_samples_leaf = 5, random_state = 5000 + k)
    rf.fit(X_train, y_train)
    rf_scores_2[k] = rf.predict_proba(X)[:, 1]

lr_vendor_score_2 = lr_scores_2.mean(axis = 0)
rf_vendor_score_2 = rf_scores_2.mean(axis = 0)

lr_rho, _ = stats.spearmanr(lr_vendor_score, lr_vendor_score_2)
rf_rho, _ = stats.spearmanr(rf_vendor_score, rf_vendor_score_2)
print(f"LR Spearman rho between two runs: {lr_rho:.4f}")
print(f"RF Spearman rho between two runs: {rf_rho:.4f}")
print(f"\nTarget: rho > 0.90. {'Both pass.' if min(lr_rho, rf_rho) > 0.90 else 'Stability concern.'}")

LR Spearman rho between two runs: 0.9977
RF Spearman rho between two runs: 0.9991

Target: rho > 0.90. Both pass.


In [ ]:
# select the better model
if np.mean(holdout_recalls_rf) >= np.mean(holdout_recalls_lr):
    best_model = "RF"
    vendor_score = rf_vendor_score
    best_importance = importance.sort_values("rf_importance", ascending = False)
else:
    best_model = "LR"
    vendor_score = lr_vendor_score
    best_importance = importance.sort_values("lr_coef_abs", ascending = False)

print(f"Selected model: {best_model}")
print(f"Score range: [{vendor_score.min():.4f}, {vendor_score.max():.4f}]")

Selected model: RF
Score range: [0.0190, 0.9972]


## Score All TINs

In [ ]:
# assign scores and tiers
df["vendor_score"] = vendor_score
df["vendor_pctile"] = df["vendor_score"].rank(pct = True)
df["vendor_tier"] = pd.cut(
    df["vendor_pctile"],
    bins = [0, 0.70, 0.85, 0.95, 1.0],
    labels = ["Minimal", "Low", "Medium", "High"],
    include_lowest = True,
)

top_features_list = best_importance["feature"].head(10).tolist()

def get_top_features(row, features, n = 3):
    deviations = {}
    for f in features:
        pop_med = df[f].median()
        pop_std = df[f].std()
        if pop_std > 0:
            deviations[f] = abs(row[f] - pop_med) / pop_std
        else:
            deviations[f] = 0
    top = sorted(deviations, key = deviations.get, reverse = True)[:n]
    return ", ".join(top)

df["top_contributing_features"] = df.apply(lambda r: get_top_features(r, top_features_list), axis = 1)

tier_summary = df.groupby("vendor_tier", observed = True).agg(
    n_tins = ("prov_tin", "count"),
    n_cadence = ("is_cadence", "sum"),
    avg_score = ("vendor_score", "mean"),
).reset_index()

print("Tier distribution:")
print(tier_summary.to_string(index = False))

Tier distribution:
vendor_tier  n_tins  n_cadence  avg_score
    Minimal    5077          1   0.182915
        Low    1088          6   0.368239
     Medium     726          7   0.481324
       High     363        106   0.734702


In [ ]:
# export
output_dir = r"C:\Users\knguy139\Documents\Projects\Data\Output"

export_cols = id_cols + all_features + ["vendor_score", "vendor_tier", "top_contributing_features"]
df[export_cols].to_csv(f"{output_dir}\\kn_rpm_cadence_scored.csv", index = False)
print(f"Exported {len(df)} rows to kn_rpm_cadence_scored.csv")

baseline_export = baseline.merge(
    importance[["feature", "lr_coef_mean", "lr_rank", "rf_importance", "rf_rank"]],
    on = "feature", how = "left"
)
baseline_export.to_csv(f"{output_dir}\\kn_rpm_cadence_feature_summary.csv", index = False)
print(f"Exported {len(baseline_export)} features to kn_rpm_cadence_feature_summary.csv")

print("\nTop 20 non-Cadence TINs by vendor_score:")
top_non_cadence = (
    df[df["is_cadence"] == 0]
    .nlargest(50, "vendor_score")
    [["prov_tin", "hospital_group", "total_mbrs", "vendor_score", "vendor_tier", "top_contributing_features"]]
)
print(top_non_cadence.to_string(index = False))

Exported 7254 rows to kn_rpm_cadence_scored.csv
Exported 38 features to kn_rpm_cadence_feature_summary.csv

Top 20 non-Cadence TINs by vendor_score:
 prov_tin                          hospital_group  total_mbrs  vendor_score vendor_tier                                 top_contributing_features
810745237                  Lutheran Health System          81      0.989023        High              distinct_srvc_npis, pct_hf, pct_device_lines
452535719      KNOXVILLE HMA PHYSICIAN MANAGEMENT         388      0.988447        High      distinct_srvc_npis, pct_device_lines, pct_mgmt_lines
208445687                  Lutheran Health System          45      0.988187        High      pct_device_lines, distinct_srvc_npis, pct_mgmt_lines
621681750                    NORTHWEST PHYSICIANS          54      0.986079        High      distinct_srvc_npis, pct_device_lines, pct_mgmt_lines
264214938                  Lutheran Health System         162      0.977410        High              pct_hf, distinct_srv

In [ ]:
# checkpoint
import pickle

checkpoint = {
    "df": df,
    "X": X,
    "y": y,
    "model_features": model_features,
    "scaler": scaler,
    "lr_vendor_score": lr_vendor_score,
    "rf_vendor_score": rf_vendor_score,
    "baseline": baseline,
    "importance": importance,
    "best_model": best_model,
}

with open(r"C:\Users\knguy139\Documents\Projects\RPM\rpm_cadence_checkpoint.pkl", "wb") as f:
    pickle.dump(checkpoint, f)
print("Checkpoint saved.")

engine.dispose()
print("Done.")

Checkpoint saved.
Done.
